# সহজ ভাষায় Notebook Guide

এই notebook-এ Neural Network-এর theory এবং code পাশাপাশি শেখানো হয়েছে। Technical term English-এ থাকবে, আর explanation Bangla-তে থাকবে।

## কীভাবে ব্যবহার করবেন?

1. Cell উপর থেকে নিচে sequence অনুযায়ী run করুন।
2. প্রতিটি output-এর shape, value এবং graph-এর meaning লক্ষ্য করুন।
3. Graph-এর ঠিক পরের explanation পড়ে axis, color, boundary এবং metric বুঝুন।
4. Error হলে আগে project path, import এবং data shape check করুন।

> সব cell run হলেই result correct প্রমাণ হয় না; learning behavior এবং expected pattern-ও validate করতে হবে।

# Class 13 — Neural Networks from Scratch
## Hands-On Lab: Neuron, Activation, Forward Pass & Backpropagation

এই lecture-এ আমরা NumPy ব্যবহার করে একটি ছোট neural network বুঝব। লক্ষ্য হলো black-box API মুখস্থ না করে matrix multiplication, activation এবং gradient descent-এর flow দেখা।

### Learning Objectives

- একটি single neuron-এর weighted sum এবং bias বুঝতে পারা
- Sigmoid, ReLU এবং Tanh-এর shape ও derivative compare করা
- 2-layer network-এর forward pass trace করা
- Backpropagation ও gradient descent দিয়ে XOR শেখানো
- Loss curve এবং decision boundary সঠিকভাবে পড়া

In [ ]:
import sys
from pathlib import Path
import warnings

PROJECT_ROOT = Path.cwd().parent / "Class 1 Project"
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd() / "week 7" / "Class 1 Project"
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.core.neuron import Neuron
from src.core.activation import ActivationFunctions
from src.core.network import NeuralNetwork
from src.gates.logic_gates import LogicGateDataset
from src.visualization.nn_viz import NetworkVisualizer

warnings.filterwarnings("ignore")
np.set_printoptions(precision=3, suppress=True)
print("Imports ready:", PROJECT_ROOT)

## Part 1: The Neuron

একটি neuron প্রথমে input ও weight-এর dot product করে, তারপর bias যোগ করে:

[
z = w cdot x + b
]

এই z হলো activation-এর আগের signal। Weight input-এর গুরুত্ব এবং bias decision threshold shift করে।

In [ ]:
# একটি neuron-এর weighted sum trace করি
neuron = Neuron(weights=[0.5, -0.3], bias=0.1)
inputs = [1.0, 2.0]

z = neuron.compute_z(inputs)
activation = ActivationFunctions()
print(f"Inputs: {inputs}")
print(f"Weights: {neuron.weights}")
print(f"Bias: {neuron.bias}")
print(f"Weighted sum z = {z:.4f}")
print(f"Sigmoid(z) = {activation.sigmoid(np.array([z]))[0]:.4f}")
print(f"ReLU(z) = {activation.relu(np.array([z]))[0]:.4f}")

fig = NetworkVisualizer.draw_single_neuron(inputs, neuron.weights, neuron.bias, z)
plt.show()

### Graph Explanation — Single Neuron

এই diagram-এ নীল circle হলো input, purple label হলো weight, হলুদ circle হলো bias, আর মাঝের box weighted sum তৈরি করে।

- Arrow-এর thickness weight-এর magnitude বোঝায়; negative weight-এর sign আলাদা করে text-এ দেখা যায়।
- Box-এর z activation-এর আগের value; output node শুধু final signal-এর ধারণা দেখায়।
- Input বদলালে একই formula দিয়ে z বদলাবে। তাই neuron হলো neural network-এর smallest computation unit।

## Part 2: Activation Functions

Activation function network-এ non-linearity যোগ করে। Non-linearity না থাকলে অনেক layer-ও শেষ পর্যন্ত একটি linear transformation-এর মতো আচরণ করবে।

আমরা Sigmoid, ReLU এবং Tanh-এর function ও derivative একসাথে দেখব।

In [ ]:
x = np.linspace(-6, 6, 400)
act = ActivationFunctions()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, act.sigmoid(x), label="Sigmoid")
axes[0].plot(x, act.relu(x), label="ReLU")
axes[0].plot(x, act.tanh(x), label="Tanh")
axes[0].set(title="Activation functions", xlabel="Input z", ylabel="Output")
axes[0].legend()

axes[1].plot(x, act.sigmoid_derivative(x), label="Sigmoid derivative")
axes[1].plot(x, act.relu_derivative(x), label="ReLU derivative")
axes[1].plot(x, act.tanh_derivative(x), label="Tanh derivative")
axes[1].set(title="Derivatives used in learning", xlabel="Input z", ylabel="Derivative")
axes[1].legend()

plt.tight_layout()
plt.show()

### Graph Explanation — Activation Functions

বাম graph-এ x-axis হলো pre-activation z এবং y-axis হলো function output।

- Sigmoid-এর output 0 থেকে 1-এর মধ্যে, তাই binary probability output-এ useful।
- ReLU negative z-কে 0 করে এবং positive z-কে রেখে দেয়; hidden layer-এ এটি fast এবং common।
- Tanh-এর output -1 থেকে 1, তাই এটি zero-centered।

ডান graph-এ derivative হলো slope বা learning signal। Saturated Sigmoid/Tanh অঞ্চলে derivative ছোট, তাই সেখানে gradient ধীরে চলে। ReLU-তে negative অংশের derivative 0—এটাই dying ReLU limitation।

## Part 3: Forward Pass Through a 2-Layer Network

এখন একটি network দেখি:

[
X ightarrow W_1,b_1 ightarrow ReLU ightarrow W_2,b_2 ightarrow Sigmoid ightarrow hat{y}
]

Input shape এবং weight shape মিলিয়ে matrix multiplication করা হয়।

In [ ]:
nn = NeuralNetwork(input_size=2, hidden_size=3, output_size=1, seed=42)
X_one = np.array([[1.0, 0.0]])
prediction = nn.forward(X_one)

print("W1 shape:", nn.W1.shape)
print("b1 shape:", nn.b1.shape)
print("W2 shape:", nn.W2.shape)
print("b2 shape:", nn.b2.shape)
print("Hidden activation a1:", nn.cache["a1"].round(3))
print("Prediction y_hat:", prediction.round(4))

fig = NetworkVisualizer.draw_network_architecture(2, 3, 1)
plt.show()

### Graph Explanation — Network Architecture

এখানে বাম layer-এ 2টি input, মাঝের layer-এ 3টি hidden neuron এবং ডান layer-এ 1টি output আছে।

- Input থেকে hidden connection-এর weight matrix হলো W1, তাই W1-এর shape 3 × 2।
- Hidden থেকে output connection-এর matrix হলো W2, তাই W2-এর shape 1 × 3।
- Hidden layer ReLU এবং output layer Sigmoid ব্যবহার করছে।
- Architecture diagram structure বোঝায়; কোনো node-এর learned value জানতে output cell-এর printed cache দেখতে হবে।

## Part 4: Backpropagation দিয়ে XOR শেখানো

XOR হলো non-linearly separable problem:

| x1 | x2 | Target |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Forward pass prediction তৈরি করে। Loss ভুল মাপে। Backward pass chain rule দিয়ে gradient বের করে, তারপর gradient descent weight update করে।

In [ ]:
dataset = LogicGateDataset("XOR")
X, y = dataset.get_data()

model = NeuralNetwork(
    input_size=2,
    hidden_size=4,
    output_size=1,
    learning_rate=0.5,
    seed=42,
)
history = model.train(X, y, epochs=5000, verbose=False)
predictions = model.predict(X)
accuracy = np.mean(np.round(predictions.flatten()) == y.flatten())

print("Rounded predictions:", np.round(predictions.flatten()).astype(int))
print(f"Initial loss: {history['loss'][0]:.4f}")
print(f"Final loss: {history['loss'][-1]:.4f}")
print(f"Accuracy: {accuracy:.0%}")
assert accuracy >= 0.75

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history["loss"], color="#6C5CE7")
ax.set(title="XOR training loss", xlabel="Epoch", ylabel="Binary cross-entropy loss")
plt.tight_layout()
plt.show()

### Graph Explanation — Loss Curve

এই graph-এ x-axis হলো epoch এবং y-axis হলো Binary Cross-Entropy loss।

- Curve নিচে নামা মানে prediction target-এর কাছাকাছি হচ্ছে।
- শুরুতে loss বেশি, training-এর পরে loss কম—এটি learning-এর evidence।
- Loss একেবারে 0 না হলেও rounded prediction ঠিক হতে পারে; তাই loss-এর সাথে accuracy-ও দেখা দরকার।
- Curve আটকে গেলে learning rate, hidden size, initialization এবং activation check করতে হয়।

In [ ]:
fig = NetworkVisualizer.plot_decision_boundary(
    model, X, y, title="XOR", resolution=0.02
)
plt.show()

### Graph Explanation — XOR Decision Boundary

এই visualization-এর background color হলো model-এর predicted probability; color transition-এর মাঝের dashed line হলো 0.5 decision boundary।

- চারটি dot হলো XOR truth-table-এর four input combinations।
- Dot-এর color target label বোঝায়; background model কী predict করছে তা বোঝায়।
- XOR-এ একটি straight line যথেষ্ট নয়। Hidden layer non-linearity ব্যবহার করে দুই class-এর জন্য curved/segmented region তৈরি করে।
- Boundary-এর বাইরে থাকা কোনো point দেখলে নতুন input-এ model-এর probability কীভাবে বদলায় তা বোঝা যায়।

## Summary Checklist

- [ ] Neuron: (z = w cdot x + b)
- [ ] Weight input-এর influence বদলায়; bias threshold shift করে।
- [ ] Activation function non-linearity যোগ করে।
- [ ] Forward pass prediction তৈরি করে।
- [ ] Backpropagation gradient বের করে; gradient descent weight update করে।
- [ ] Hidden layer XOR-এর মতো non-linear pattern শেখাতে সাহায্য করে।

### Final Reflection

নিজের ভাষায় লিখুন: XOR single neuron দিয়ে কেন কঠিন, hidden layer কী বদলায়, এবং শুধু accuracy দেখে model ভালো বলা কেন যথেষ্ট নয়?